# H-Reflex Post-Hoc Analysis

Loads peri-stimulus trial files from the H-Reflex App and runs the full post-hoc analysis pipeline.

**File convention (V2/V3):**
- **`.hrs1`** — MH Recruitment Curve stage (sweeps intensities)
- **`.hrs2`** — Control Mode stage (fixed intensity, user-adjustable)
- **`.hrs3`** — Down Condition Pellet (DCP) stage
- **`.hrs4`** — Up Condition Pellet stage
- **`.hrs5`** — Down Condition VNS stage
- **`.hrs6`** — Up Condition VNS stage

# Section 1: Setup

Import all analysis utilities from `helpers.py`.

In [ ]:
import os
import numpy as np
from helpers import (
    read_hrs2, read_hrs3, read_hrs4, read_hrs5, read_hrs6, read_hrs_ft,
    find_hrs_files, detect_app_version,
    print_hrs2_summary,
    plot_amplitude_distribution, plot_actual_trial_timeline,
    filter_failed_trials,
    plot_hrs2_analysis, plot_hrs2_trials,
    plot_background_grand_means,
    compute_trial_bins, filter_trials, compute_equal_bin_ranges,
    build_merged_amp_groups,
    plot_bin_overview, create_bin_viewer, print_bin_statistics,
    split_trials_by_polarity, plot_hm_ratio_summary,
    compute_snr_analysis, compute_mra_snr_analysis,
    SAMPLE_RATE, BIN_DURATION_MS, BIN_SAMPLES, TRIAL_RECORD_MS,
    STIM_ONSET_THRESHOLD, STIM_END_THRESHOLD,
)

print("Helpers loaded.")
print(f"  App constants: SAMPLE_RATE={SAMPLE_RATE} Hz | BIN={BIN_DURATION_MS} ms ({BIN_SAMPLES} samples) | TRIAL_RECORD={TRIAL_RECORD_MS} ms")
print(f"  Stim thresholds: onset >= {STIM_ONSET_THRESHOLD} V | end < {STIM_END_THRESHOLD} V")

# Section 2: Recording Files

Set `recording_dir` to the folder containing your `.hrs1` / `.hrs2` / `.hrs3` files  
(`V2`: `.hrs1`=MH Recruitment, `.hrs2`=Control Mode, `.hrs3`=DCP).  
Set `recording_sample_rate` to override the sample rate; `None` auto-detects it from the HRS1 header.

In [ ]:
recording_dir         = "1"
recording_sample_rate = None  # set to e.g. 5000 or 10000 to override; None = auto-detect

hrs1_path, hrs2_path, hrs3_path, hrs4_path, hrs5_path, hrs6_path, hrsft_path = find_hrs_files(recording_dir)
_app_version = detect_app_version(recording_dir)
print(f"H-Reflex App V{_app_version}  |  "
      f"hrs1={os.path.basename(hrs1_path) if hrs1_path else chr(8211)}  "
      f"hrs2={os.path.basename(hrs2_path) if hrs2_path else chr(8211)}  "
      f"hrs3={os.path.basename(hrs3_path) if hrs3_path else chr(8211)}  "
      f"hrs4={os.path.basename(hrs4_path) if hrs4_path else chr(8211)}  "
      f"hrs5={os.path.basename(hrs5_path) if hrs5_path else chr(8211)}  "
      f"hrs6={os.path.basename(hrs6_path) if hrs6_path else chr(8211)}  "
      f"hrsft={os.path.basename(hrsft_path) if hrsft_path else chr(8211)}")

# Section 3: Read Data Files

Load the binary recording files and print summaries.

`.hrs1` = MH Recruitment Curve, `.hrs2` = Control Mode, `.hrs3` = DCP.

In [ ]:
# --- Load all data files (V2/V3 app) ------------------------------------------
cm_header  = cm_trials  = cm_emg_blocks  = None
dcp_header = dcp_trials = dcp_emg_blocks = None
s4_header  = s4_trials  = s4_emg_blocks  = None
s5_header  = s5_trials  = s5_emg_blocks  = None
s6_header  = s6_trials  = s6_emg_blocks  = None
ft_header  = ft_trials  = ft_emg_blocks  = None

# V2/V3: .hrs1 = MH Recruitment Curve, .hrs2 = Control Mode, .hrs3+ = conditioning stages
if hrs1_path:
    hrs2_header, hrs2_trials, hrs2_emg_blocks = read_hrs2(hrs1_path)
    hrs1_header = hrs2_header
    print_hrs2_summary(hrs2_header, hrs2_trials, hrs2_emg_blocks, hrs1_path)
else:
    print("No .hrs1 file found — MH Recruitment data unavailable.")
    hrs1_header = hrs2_header = None
    hrs2_trials = []
    hrs2_emg_blocks = []
if hrs2_path:
    cm_header, cm_trials, cm_emg_blocks = read_hrs2(hrs2_path)
    print(f"Control Mode:           {len(cm_trials)} trials")
else:
    print("No .hrs2 file found — Control Mode data unavailable.")
if hrs3_path:
    dcp_header, dcp_trials, dcp_emg_blocks = read_hrs3(hrs3_path)
    print(f"Down Condition Pellet:  {len(dcp_trials)} trials")
else:
    print("No .hrs3 file found — Down Condition Pellet data unavailable.")
if _app_version >= 3:
    if hrs4_path:
        s4_header, s4_trials, s4_emg_blocks = read_hrs4(hrs4_path)
        print(f"Up Condition Pellet:    {len(s4_trials)} trials")
    else:
        print("No .hrs4 file found — Up Condition Pellet data unavailable.")
    if hrs5_path:
        s5_header, s5_trials, s5_emg_blocks = read_hrs5(hrs5_path)
        print(f"Down Condition VNS:     {len(s5_trials)} trials")
    else:
        print("No .hrs5 file found — Down Condition VNS data unavailable.")
    if hrs6_path:
        s6_header, s6_trials, s6_emg_blocks = read_hrs6(hrs6_path)
        print(f"Up Condition VNS:       {len(s6_trials)} trials")
    else:
        print("No .hrs6 file found — Up Condition VNS data unavailable.")
    if hrsft_path:
        ft_header, ft_trials, ft_emg_blocks = read_hrs_ft(hrsft_path)
        print(f"Frequency Test:         {len(ft_trials)} trials")
    else:
        print("No .hrsft file found — Frequency Test data unavailable.")

# Control Mode-only: alias as primary analysis when no MH Recruitment stage
if not hrs2_trials and cm_trials:
    hrs2_header     = cm_header
    hrs2_trials     = cm_trials
    hrs2_emg_blocks = cm_emg_blocks
    hrs1_header     = hrs2_header
    print('Note: Control Mode trials used as primary analysis (no Recruitment Curve stage).')

# Ensure hrs1_header.sample_rate is always accessible for downstream cells
if hrs1_header is None:
    class _SampleRateStub:
        sample_rate = recording_sample_rate or 5000.0
    hrs1_header = _SampleRateStub()

# Section 4: Analysis Configuration

Set the M-wave and H-wave detection windows and viewing parameters used by all downstream sections.  
**Change these here** — every subsequent section picks them up automatically.

In [ ]:
# Viewing windows (ms relative to stim onset)
PRE_PLOT_MS  = 5
POST_PLOT_MS = 25
PRE_AVG_MS   = 5
POST_AVG_MS  = 25
N_PER_PAGE   = 6

# M/H wave detection windows (ms relative to stim onset)
M_WAVE_START_MS =  1.8
M_WAVE_END_MS   =  4.5
H_WAVE_START_MS =  7
H_WAVE_END_MS   = 12

print(f"M-wave window : {M_WAVE_START_MS} – {M_WAVE_END_MS} ms")
print(f"H-wave window : {H_WAVE_START_MS} – {H_WAVE_END_MS} ms")
print(f"Plot window   : -{PRE_PLOT_MS} to +{POST_PLOT_MS} ms  |  Avg window: -{PRE_AVG_MS} to +{POST_AVG_MS} ms")

# Section 5: Data Overview

Histogram of stimulation amplitudes delivered across the session and the actual inter-trial interval (ITI) timeline.

In [ ]:
plot_amplitude_distribution(hrs2_trials, hrs2_header)
plot_actual_trial_timeline(hrs2_trials, header=hrs2_header)

# Section 6: Trial Quality Filter

Automatically detect and remove trials where the ADC sync pulse was missed or the stim onset could not be verified.  
The cleaned trial list replaces `hrs2_trials` for all downstream sections.

In [ ]:
hrs2_trials = filter_failed_trials(
    hrs2_trials, hrs2_header, hrs2_emg_blocks,
    pre_ms=PRE_PLOT_MS, post_ms=POST_PLOT_MS,
    m_start_ms=M_WAVE_START_MS, m_end_ms=M_WAVE_END_MS,
    h_start_ms=H_WAVE_START_MS, h_end_ms=H_WAVE_END_MS,
    sample_rate=recording_sample_rate or hrs1_header.sample_rate,
)

# Section 7: Waveform Analysis

Grand-averaged EMG waveforms and recruitment curve across all stimulation amplitudes.  
A polarity toggle appears automatically for dual-polarity sessions.

In [ ]:
# ── Optional: Merged Amplitude Group Analysis ─────────────────────────────
# Pool multiple stimulation intensities into one merged group.
# MERGED_GROUPS is a list of lists, e.g.:
#   [[0.12, 0.13], [0.15, 0.16, 0.17]]  →  two merged groups
# Leave as [] to use the default (unmerged) grouping.
MERGED_GROUPS = []

if MERGED_GROUPS:
    _hrs2_trials_plot = build_merged_amp_groups(hrs2_trials, MERGED_GROUPS)
else:
    _hrs2_trials_plot = hrs2_trials


In [ ]:
plot_hrs2_analysis(
    _hrs2_trials_plot, hrs2_header,
    pre_avg_ms=PRE_AVG_MS, post_avg_ms=POST_AVG_MS,
    n_per_page=N_PER_PAGE,
    m_start_ms=M_WAVE_START_MS, m_end_ms=M_WAVE_END_MS,
    h_start_ms=H_WAVE_START_MS, h_end_ms=H_WAVE_END_MS,
    sample_rate=recording_sample_rate or hrs1_header.sample_rate,
    emg_blocks=hrs2_emg_blocks,
)

# Section 8: Individual Trial Viewer

Interactive per-trial grid with page navigation and zoom.  
Use the signal overlays to inspect the ADC sync pulse and stimulator output for each trial.  
Double-click a subplot to zoom in; use the dropdown and **View trial** button to navigate.

In [ ]:
plot_hrs2_trials(
    _hrs2_trials_plot, hrs2_header,
    pre_plot_ms=PRE_PLOT_MS, post_plot_ms=POST_PLOT_MS,
    n_per_page=N_PER_PAGE,
    m_start_ms=M_WAVE_START_MS, m_end_ms=M_WAVE_END_MS,
    h_start_ms=H_WAVE_START_MS, h_end_ms=H_WAVE_END_MS,
    sample_rate=recording_sample_rate or hrs1_header.sample_rate,
    emg_blocks=hrs2_emg_blocks,
)

# Section 9: Background EMG Distribution

Extract each trial's pre-stimulus background EMG grand mean and plot the distribution.  
For HRS2 v5+ files this uses the value stored directly in each trial; older files reconstruct it from the EMG blocks.  
The returned `state` dict (key `'trial_bg_gm'`) is passed to Section 10 for binning.

In [ ]:
state = plot_background_grand_means(
    hrs2_trials, hrs2_emg_blocks, hrs2_header,
    sample_rate=recording_sample_rate or hrs1_header.sample_rate,
)

# Section 10: EMG Activity Bins

**Optional subset filter** — restrict the analysis to a specific polarity and/or stimulation intensity
range. Set `FILTER_POLARITY` and/or `FILTER_INTENSITIES`, or leave both as `None` to include all trials.

**Bin definition** — set `BIN_RANGES = 'auto'` to automatically compute equal-count bin boundaries
from the filtered trial set, controlled by `N_BINS`. Or set `BIN_RANGES` to a list of manual `(lo, hi)` pairs.

`BIN_MODE` controls what metric is used for binning:
- `'EMG'` — pre-stim background grand mean (default)
- `'M_WAVE'` — M-wave MRA
- `'H_WAVE'` — H-wave MRA

In [ ]:
# ── Subset filter (optional) ──────────────────────────────────────────
# FILTER_POLARITY  : 'normal' | 'reversed' | None  (None = all trials)
# FILTER_POLARITY = None
FILTER_POLARITY = None

# FILTER_INTENSITIES: None (all) | tuple (lo, hi) in mA | list of exact mA values
#   e.g.  (0.5, 2.0)   keeps trials with amplitude in [0.5, 2.0] mA
#   e.g.  [1.0, 1.5]   keeps only those exact amplitudes (±0.001 mA tolerance)
# FILTER_INTENSITIES = None

FILTER_INTENSITIES = None


#FILTER_POLARITY    = None    # 'normal' | 'reversed' | None
#FILTER_INTENSITIES = None    # (lo, hi) mA | [list of mA] | None
# ───────────────────────────────────────────────────────────────────────

# ── Bin definition ──────────────────────────────────────────────────────
BIN_MODE   = 'EMG' # can also do "H-Wave", "M-Wave", "H:M"
N_BINS     = 3       # bins to create when BIN_RANGES = 'auto'
BIN_RANGES = 'auto'  # 'auto' → equal-count | [(lo, hi), ...] → manual
#BIN_RANGES = [(50, 110), (110, 140), (140, 1000)]  # µV ranges
# ───────────────────────────────────────────────────────────────────────

analysis_trials, analysis_state = filter_trials(
    hrs2_trials, state,
    filter_polarity=FILTER_POLARITY,
    filter_intensities=FILTER_INTENSITIES,
)

if BIN_RANGES == 'auto':
    BIN_RANGES = compute_equal_bin_ranges(
        analysis_trials, BIN_MODE, N_BINS, analysis_state,
        sample_rate=recording_sample_rate or hrs1_header.sample_rate,
        pre_avg_ms=PRE_AVG_MS, post_avg_ms=POST_AVG_MS,
        m_start_ms=M_WAVE_START_MS, m_end_ms=M_WAVE_END_MS,
        h_start_ms=H_WAVE_START_MS, h_end_ms=H_WAVE_END_MS,
    )

binned_trials, bin_labels, bin_colors, trial_bg_dict, bin_unit, pol_labels, _ = compute_trial_bins(
    analysis_trials,
    bin_mode=BIN_MODE,
    bin_ranges=BIN_RANGES,
    state=analysis_state,
    sample_rate=recording_sample_rate or hrs1_header.sample_rate,
    pre_avg_ms=PRE_AVG_MS, post_avg_ms=POST_AVG_MS,
    m_start_ms=M_WAVE_START_MS, m_end_ms=M_WAVE_END_MS,
    h_start_ms=H_WAVE_START_MS, h_end_ms=H_WAVE_END_MS,
)

# Section 11: Bin Overview

Averaged EMG waveforms for each bin overlaid on the same axes, plus M-wave MRA, H-wave MRA, and H:M ratio bar charts with mean ± SD.  
A polarity toggle appears automatically for dual-polarity sessions.

In [ ]:
plot_bin_overview(
    binned_trials, bin_labels, bin_colors, pol_labels, hrs2_header,
    sample_rate=recording_sample_rate or hrs1_header.sample_rate,
    pre_avg_ms=PRE_AVG_MS, post_avg_ms=POST_AVG_MS,
    m_start_ms=M_WAVE_START_MS, m_end_ms=M_WAVE_END_MS,
    h_start_ms=H_WAVE_START_MS, h_end_ms=H_WAVE_END_MS,
    bin_ranges=BIN_RANGES, bin_unit=bin_unit,
)

# Section 12: Per-Bin Viewer

Select a polarity and bin, then click **Load bin viewer** to inspect that trial subset in detail:  
background EMG distribution, averaged waveforms + recruitment curve, and individual trial grid.

In [ ]:
create_bin_viewer(
    binned_trials, bin_labels, bin_colors, pol_labels,
    hrs2_header, hrs2_emg_blocks, trial_bg_dict,
    sample_rate=recording_sample_rate or hrs1_header.sample_rate,
    pre_plot_ms=PRE_PLOT_MS, post_plot_ms=POST_PLOT_MS,
    pre_avg_ms=PRE_AVG_MS,   post_avg_ms=POST_AVG_MS,
    n_per_page=N_PER_PAGE,
    m_start_ms=M_WAVE_START_MS, m_end_ms=M_WAVE_END_MS,
    h_start_ms=H_WAVE_START_MS, h_end_ms=H_WAVE_END_MS,
)

# Section 13: Bin Statistics

Per-bin × per-polarity numeric table: trial count, mean M-wave MRA, mean H-wave MRA, and mean H:M ratio.

In [ ]:
print_bin_statistics(
    binned_trials, bin_labels, pol_labels,
    sample_rate=recording_sample_rate or hrs1_header.sample_rate,
    pre_avg_ms=PRE_AVG_MS, post_avg_ms=POST_AVG_MS,
    m_start_ms=M_WAVE_START_MS, m_end_ms=M_WAVE_END_MS,
    h_start_ms=H_WAVE_START_MS, h_end_ms=H_WAVE_END_MS,
)

# Section 14: H:M Ratio Summary

Box plot and histogram of the H:M ratio split by stimulation polarity group.  
Shows distribution shape, SD, CV, and 20th/80th percentiles.

In [ ]:
plot_hm_ratio_summary(
    split_trials_by_polarity(analysis_trials), hrs2_header,
    m_start_ms=M_WAVE_START_MS, m_end_ms=M_WAVE_END_MS,
    h_start_ms=H_WAVE_START_MS, h_end_ms=H_WAVE_END_MS,
    sample_rate=recording_sample_rate or hrs1_header.sample_rate,
)

# Section 15: SNR Analysis (RMS)

Per-amplitude signal-to-noise ratio using the RMS of the M-wave and H-wave windows against a pre-stim RMS background window.

| Metric | Formula |
|---|---|
| SNR_M | RMS(M-wave) / RMS(pre-stim BG) |
| SNR_H | RMS(H-wave) / RMS(pre-stim BG) |
| SNR_H:M | (RMS_H / RMS_M) / RMS(pre-stim BG) |

Adjust `BG_PRE_MS` to change the background window length.

In [ ]:
BG_PRE_MS = 15.0  # ms of pre-stim signal used as noise baseline

snr_results = compute_snr_analysis(
    hrs2_trials, hrs2_header,
    m_start_ms=M_WAVE_START_MS, m_end_ms=M_WAVE_END_MS,
    h_start_ms=H_WAVE_START_MS, h_end_ms=H_WAVE_END_MS,
    bg_pre_ms=BG_PRE_MS,
    sample_rate=recording_sample_rate or hrs1_header.sample_rate,
)

# Section 16: SNR Analysis (MRA)

Same analysis as Section 15 but uses **Mean Rectified Average** (`mean(|signal|)`) instead of RMS — more robust to asymmetric waveforms.

| Metric | Formula |
|---|---|
| MRA-SNR_M | MRA(M-wave) / MRA(pre-stim BG) |
| MRA-SNR_H | MRA(H-wave) / MRA(pre-stim BG) |
| MRA-SNR_H:M | (MRA_H / MRA_M) / MRA(pre-stim BG) |

In [ ]:
mra_snr_results = compute_mra_snr_analysis(
    hrs2_trials, hrs2_header,
    m_start_ms=M_WAVE_START_MS, m_end_ms=M_WAVE_END_MS,
    h_start_ms=H_WAVE_START_MS, h_end_ms=H_WAVE_END_MS,
    bg_pre_ms=BG_PRE_MS,
    sample_rate=recording_sample_rate or hrs1_header.sample_rate,
)